# derived_8.4-eval-mlp-2.2 — exploit the 54-family lr6e-4/gelu region + the mixed 3-layer gelu cell + finish the 96-family debias (~1.25 h gpu_debug H100 wall)

Follow-up to `derived_8.4-eval-mlp-2.1` (an honest negative: the mixed-family 3-seed val winner `w512x512x512_d0.3_huber0.05_lr6e-4` → test R² 0.7844 did not beat 2.0's 2-seed honest single 0.7903 / val top-5 ensemble 0.8003; SWA closed with proof — 0/152 deployments under the RNG guard; val selection shown to be the bottleneck, Spearman(val, test) −0.555 (54) / −0.309 (mixed) even at 3 seeds). 2.2 is an **optimization + further parameter sweep** of the 2.1 winners' neighborhoods, temporal protocol only (no LOSO, same honest protocol as 2.0/2.1), sized to spend ~1.25 h of the 2 h `gpu_debug` H100 wall allocation (508 job-seeds at 8 workers):

1. **54-family lr6e-4 × gelu region (headline lever)** — 2.1's test-best was `w320x320_d0.3_gelu_lr6e-4` (0.7935); the untested cells: widths below 320, lr {4e-4, 8e-4}, huber δ × lr6e-4, 3-layer × lr6e-4, dropout × width.
2. **Mixed 3-layer gelu cell** — 2.1's mixed test-best was `w448x448x448_d0.3_huber0.1_gelu` (0.7940, val rank 34!); gelu was only ever tested at lr3e-4, so 2.2 grids act × depth × lr and refines the silu-512³ huber-δ × lr surface (δ {0.03, 0.08, 0.15}, lr {4e-4, 8e-4}).
3. **96-family debias + small-net convergence** — small nets hit the 400-epoch cap under-trained (best_epoch 380–395); 2.2 adds lr {4e-4, 6e-4, 8e-4} (lr6e-4 never tested for 96), huber × lr, mixup × lr, 3-layer small nets, and max_epochs {500, 600} probes. Criterion: median bias²/MSE < 5 % (2.1: 13.9 %).
4. **Val-year diagnostic (NEW)** — every job saves best-val predictions (`val_preds.npy`) + `artifacts/val_meta.npz`; `analyze_val_years.py` computes per-config val-2021 vs val-2022 RMSE, per-year Spearman vs test, and winner stability under val-year-drop. Diagnostic only — the selection rule stays 3-seed mean val RMSE (protocol unchanged).
5. **Densest seed coverage yet** — phase-2/3 top-Ns are capped at the family sizes (66/57/82 phase-2; 42/26/40 phase-3), the direct mitigation for 2.1's val-seed-noise finding; the champion step gets per-family top-N (`sweep.champion_top_n`: mixed top-2 + 54 top-1 + 96 top-1), fixing 2.1's documented "top-2-mixed not expressible" limitation.
6. **No SWA re-spend** — SWA is a closed negative; no SWA configs run in 2.2. `fg`/`plr` stay closed negatives.

Protocol unchanged and honest: train on train (2017–2020, n=9,803), early-stop/select on official val (2021–2022, n=4,805), evaluate on untouched test (2023–2025, n=6,620); multi-seed mean val RMSE selection among the mlp/fg/plr winner pool (mlp-only in practice); aux2020 diagnostic only; patience-60 kept; no calibration / no trainval retrain (documented negatives). LOSO out of scope (same protocol as 2.0/2.1).

All numbers below are the stdout of this executed notebook. Weights/checkpoints/test predictions under `models/`; preprocessed tensors and per-job logs under `artifacts/`; figures at the experiment root.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-2.2", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
df_bias = pd.read_csv(EXP_DIR / "bias_summary.csv") if (EXP_DIR / "bias_summary.csv").exists() else None
df_bias_cl = pd.read_csv(EXP_DIR / "bias_by_cluster.csv") if (EXP_DIR / "bias_by_cluster.csv").exists() else None
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv") if (EXP_DIR / "ood_summary.csv").exists() else None
df_stop21 = pd.read_csv(EXP_DIR / "stopping_21_summary.csv") if (EXP_DIR / "stopping_21_summary.csv").exists() else None
df_stop21_agg = pd.read_csv(EXP_DIR / "stopping_21_aggregate.csv") if (EXP_DIR / "stopping_21_aggregate.csv").exists() else None
df_sel = pd.read_csv(EXP_DIR / "selection_summary.csv") if (EXP_DIR / "selection_summary.csv").exists() else None
df_valyears = pd.read_csv(EXP_DIR / "val_year_summary.csv") if (EXP_DIR / "val_year_summary.csv").exists() else None
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

fam_labels = {"2regime_96": "2-Regime-96", "2regime_54": "2-Regime-54", "2regime_mixed": "2-Regime-Mixed"}
print("loaded", len(df_summary), "leaderboard rows,", len(df_sweep), "sweep rows")

loaded 69 leaderboard rows, 207 sweep rows


## Selection Protocol v9 Diagnostic

Selection = multi-seed mean val RMSE among the honest architectures (mlp / fg / plr — 2.2 runs only mlp configs, so the pool is mlp-only in practice). 2.2 keeps the 3-phase, 3-seed sweep {42, 7, 123} from 2.1 with the **densest coverage yet** (phase-2/3 top-Ns capped at the family sizes) because 2.1 documented that the mixed/54 families' val ranking is noisy even at 3 seeds (Spearman(val, test) = -0.309 / -0.555). aux2020 stays diagnostic-only (measures train fit). This section reports the val ranking, the Spearman correlations vs test at 1-/2-/3-seed aggregation, and the phase-stability table from `analyze_selection.py`.

In [2]:
from scipy.stats import spearmanr
HONEST = ("mlp", "fg", "plr")
print("### Selection Protocol v9 Diagnostic (selection = multi-seed mean val RMSE; mlp/fg/plr pool)")
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].dropna(subset=["test_r2"]).copy()
    if sub.empty:
        continue
    sub = sub.sort_values("val_rmse", na_position="last").reset_index(drop=True)
    print(f"\n#### {fam_label} — top-10 by val RMSE")
    cols = ["config_id", "architecture", "n_seeds", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias"]
    print(sub.head(10)[cols].to_markdown(index=False))
    valid = sub.dropna(subset=["val_rmse", "test_r2"])
    if len(valid) >= 8:
        rho, p = spearmanr(valid["val_rmse"], valid["test_r2"])
        print(f"  Spearman(val_rmse, test_r2) = {rho:+.3f} (p={p:.3f}, n={len(valid)})")
    honest_sub = sub[sub["architecture"].isin(HONEST)]
    val_honest = honest_sub.sort_values("val_rmse").iloc[0]
    test_best = sub.sort_values("test_r2", ascending=False).iloc[0]
    print(f"  val winner (honest) : {val_honest['config_id']} (test_r2={val_honest['test_r2']:.4f})")
    print(f"  test best (ref)     : {test_best['config_id']} (test_r2={test_best['test_r2']:.4f})")

if df_sel is not None:
    print("\n### Selection-reliability summary (analyze_selection.py)")
    print("#### Spearman(val, test) by aggregation depth")
    sub = df_sel[df_sel["aggregation"].str.startswith(("1-seed", "2-seed", "3-seed"))]
    print(sub.to_markdown(index=False))
    print("\n#### Phase stability — winner at each seed depth")
    print(df_sel[df_sel["aggregation"].str.startswith("winner|")].to_markdown(index=False))

### Selection Protocol v9 Diagnostic (selection = multi-seed mean val RMSE; mlp/fg/plr pool)

#### 2-Regime-96 — top-10 by val RMSE
| config_id                     | architecture   |   n_seeds |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |
|:------------------------------|:---------------|----------:|-----------:|-----------:|----------:|------------:|------------:|
| w512x512x512_d0.3_lr1e-3      | mlp            |         3 |  0.0484652 |  0.0260292 |  0.759483 |   0.0499584 |   0.0188537 |
| w320x320_d0.5_huber0.1_lr1e-3 | mlp            |         3 |  0.0493431 |  0.0300213 |  0.729662 |   0.0529651 |   0.0280154 |
| w320x320_d0.5_huber0.2_lr1e-3 | mlp            |         3 |  0.0497279 |  0.031144  |  0.716225 |   0.0542654 |   0.0302069 |
| w320x320_d0.5_huber0.1_lr6e-4 | mlp            |         3 |  0.0497972 |  0.032205  |  0.724091 |   0.053508  |   0.0286928 |
| w320x320_d0.5_huber0.2_lr6e-4 | mlp            |         3 |  0.0504033 |  0.0321439 |  0.71

## Val-Year Selection Reliability (NEW in 2.2)

2.1's headline negative: the mixed/54 families' val ranking is noisy even at 3-seed aggregation (Spearman(val, test) = -0.309 / -0.555), and more seeds on the same val split did not fix it. This diagnostic splits the official val set (2021–2022) by YEAR and asks which val year is the better proxy for test, and whether the val-selected winner is stable under leave-one-val-year-out selection. It is made possible by the new `val_preds.npy` (best-val predictions per job, saved by the mlp22 trainer) + `artifacts/val_meta.npz`. **Diagnostic only** — the deployed selection rule stays 3-seed mean val RMSE on the FULL official val (protocol unchanged). Backed by `analyze_val_years.py` (`val_year_summary.csv`).

In [3]:
from scipy.stats import spearmanr as _spearmanr
if df_valyears is not None:
    print("### Val-year diagnostic (val-2021 vs val-2022; 3-seed mean val RMSE per config)")
    for family, fam_label in fam_labels.items():
        sub = df_valyears[df_valyears["family"] == family].sort_values("val_rmse", na_position="last").head(10)
        if sub.empty:
            continue
        print(f"\n#### {fam_label} — top-10 by full-val RMSE (with per-year val RMSE)")
        cols = ["config_id", "n_seeds", "val_rmse", "val_2021_rmse", "val_2022_rmse", "test_r2"]
        print(sub[cols].to_markdown(index=False))

    print("\n### Spearman(val signal, test R2) per family (phase-1 pool, 3-seed aggregation)")
    rows = []
    for family in fam_labels:
        sub = df_valyears[df_valyears["family"] == family].dropna(subset=["val_rmse", "test_r2"])
        for sig in ("val_rmse", "val_2021_rmse", "val_2022_rmse"):
            s = sub.dropna(subset=[sig])
            if len(s) >= 8:
                rho, p = _spearmanr(s[sig], s["test_r2"])
                rows.append({"family": family, "signal": sig, "n_configs": len(s),
                             "spearman": f"{rho:+.3f}", "p_value": f"{p:.4f}"})
    print(pd.DataFrame(rows).to_markdown(index=False))

    print("\n### Winner stability under leave-one-val-year-out selection (3-seed means)")
    rows = []
    for family in fam_labels:
        sub = df_valyears[df_valyears["family"] == family].dropna(subset=["val_rmse"])
        if sub.empty:
            continue
        for sig in ("val_rmse", "val_2021_rmse", "val_2022_rmse"):
            s = sub.dropna(subset=[sig]).sort_values(sig)
            if s.empty:
                continue
            w = s.iloc[0]
            rows.append({"family": family, "selected_by": sig,
                         "winner": w["config_id"], "winner_test_r2": f"{w['test_r2']:.4f}"})
    print(pd.DataFrame(rows).to_markdown(index=False))
else:
    print("val_year_summary.csv not found — run analyze_val_years.py after the sweep.")

### Val-year diagnostic (val-2021 vs val-2022; 3-seed mean val RMSE per config)

#### 2-Regime-96 — top-10 by full-val RMSE (with per-year val RMSE)
| config_id                     |   n_seeds |   val_rmse |   val_2021_rmse |   val_2022_rmse |   test_r2 |
|:------------------------------|----------:|-----------:|----------------:|----------------:|----------:|
| w512x512x512_d0.3_lr1e-3      |         3 |  0.0484652 |       0.0403419 |       0.0557868 |  0.759483 |
| w320x320_d0.5_huber0.1_lr1e-3 |         3 |  0.0493431 |       0.0411788 |       0.0567137 |  0.729662 |
| w320x320_d0.5_huber0.2_lr1e-3 |         3 |  0.0497279 |       0.0412356 |       0.057361  |  0.716225 |
| w320x320_d0.5_huber0.1_lr6e-4 |         3 |  0.0497972 |       0.0413    |       0.0574368 |  0.724091 |
| w320x320_d0.5_huber0.2_lr6e-4 |         3 |  0.0504033 |       0.0411828 |       0.0585985 |  0.715018 |
| w128x128x128_d0.4_lr1e-3      |         3 |  0.0504319 |       0.040164  |       0.059393  |  0.7286

## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id` and `n_seeds`; `(val top-k avg)` rows are offline seed-averaged ensembles of the top-k val-selected honest configs (no extra training); `(5-seed champ, ...)` rows are 5-seed champion ensembles of the val-selected winners (extra stability seeds, no trainval retrain — documented negative); `cross-family` rows average the val-selected winners across families. XGBoost rows are the eval-1.1 references; `MLP-1.3` / `MLP-2.0` / `MLP-2.1` rows are the previous experiments' val-selected winners + test-best references (2.0's mixed val top-5 ensemble 0.8003 is the number 2.2 must beat; 2.1's mixed val winner 0.7844 and test-best 0.7940 are the nearer bars); `test-best` rows are reporting only (selection on test would be leakage).

In [4]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))

### Overall Leaderboard (2023-2025 Test Set)
| model_name                                                                                                                                                                 | strategy_name          |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                                 | XGBoost_Reference      |    0.81496  |     0.0438196 |       0.043337  |    0.00648567 |    0.0337195 |         0.905594 |
| MLP-2.0 2-Regime-Mixed (val top-5 avg)                     

## Hyperparameter Sweep Summary

191 curated phase-1 configs (all `mlp` — `fg`/`plr` are documented negatives from 2.0, `swa` a documented negative from 2.1, and none get GPU) generated deterministically by `make_configs.py` from 2-factor grids around the 2.1 winners: 54 gelu/lr6e-4 width × lr fine / width × huber / 3-layer × lr / 3-layer × huber / dropout × width / dropout × huber; mixed act × depth × lr / gelu × huber × lr / silu-512³ δ × lr fine / dropout × δ / gelu 2-layer × lr / δ-0.05 probes; 96 small-net width × lr fine / huber × lr / max_epochs {500, 600} / 3-layer small nets / mixup × lr / dropout × huber. 8 parallel H100 workers; configs ranked by **multi-seed mean val RMSE** (honest signal); test R² for reference. Phase-2 configs carry `n_seeds=2`, phase-3 configs `n_seeds=3` (top-Ns capped at the family sizes — the densest seed coverage yet).

In [5]:
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].sort_values("val_rmse", na_position="last").head(10)
    if sub.empty:
        continue
    print(f"### Sweep Top-10 — {fam_label} (by val RMSE, the honest selection signal)")
    cols = ["config_id", "architecture", "n_seeds", "dropout", "lr", "loss", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias", "best_epoch", "train_time_s"]
    show = sub[cols].copy()
    show["deployed"] = [json.loads((EXP_DIR / "models" / family / cid / "meta.json").read_text()).get("deployed", "live")
                        if (EXP_DIR / "models" / family / cid / "meta.json").exists() else "" for cid in sub["config_id"]]
    print(show.to_markdown(index=False))
    print()

### Sweep Top-10 — 2-Regime-96 (by val RMSE, the honest selection signal)
| config_id                     | architecture   |   n_seeds |   dropout |     lr | loss   |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |   best_epoch |   train_time_s | deployed   |
|:------------------------------|:---------------|----------:|----------:|-------:|:-------|-----------:|-----------:|----------:|------------:|------------:|-------------:|---------------:|:-----------|
| w512x512x512_d0.3_lr1e-3      | mlp            |         3 |       0.3 | 0.001  | mse    |  0.0484652 |  0.0260292 |  0.759483 |   0.0499584 |   0.0188537 |          263 |        124.873 | live       |
| w320x320_d0.5_huber0.1_lr1e-3 | mlp            |         3 |       0.5 | 0.001  | huber  |  0.0493431 |  0.0300213 |  0.729662 |   0.0529651 |   0.0280154 |          199 |        103.297 | live       |
| w320x320_d0.5_huber0.2_lr1e-3 | mlp            |         3 |       0.5 | 0.001  | huber  |  0.0497279 |  0.0

## Per-Regime Performance Breakdown

Cluster 0 holds 73 % of the test rows, so it dominates the pooled R². Per-cluster test metrics for the val top-3 honest configs per family (the mixed family's c1 = 54+10 specialist is expected to hold the ~0.83 R² of the 54-family's c1 while c0 gains the 96-pool fit), the XGBoost references, and the 1.3 / 2.0 / 2.1 reference winners.

In [6]:
print("### Per-Regime Performance Breakdown")
cols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print(df_per_regime[cols].to_markdown(index=False))

### Per-Regime Performance Breakdown
| strategy_name     | model_name                                                |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:------------------|:----------------------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)                |         0 |      7156 |     4817 | 0.751308 | 0.0498896 | 0.0471617 |  0.0162712   | 0.0390899 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)                |         1 |      2647 |     1803 | 0.77822  | 0.0501416 | 0.0430228 |  0.0257531   | 0.0377413 |
| MLP_2regime_96    | MLP 2-Regime-96 (w320x320_d0.5_huber0.1_lr1e-3)           |         0 |      7156 |     4817 | 0.714825 | 0.0534239 | 0.0457827 |  0.0275327   | 0.043497  |
| MLP_2regime_96    | MLP 2-Regime-96 (w320x320_d0.5_huber0.1_lr1e-3

## Yearly Performance Breakdown

Year-by-year R² on the 2023–2025 test period. 2.0 fixed the historically weak 2025 year for the mixed family's ensembles (2025 R² 0.8336 — best of any model); 2.1's mixed val winner held 2025 at 0.8185. This table tracks whether the 2.2 winners hold that year.

In [7]:
year_cols = [c for c in df_summary.columns if c.startswith("year_") and c.endswith("_r2")]
print("### Year-by-Year R² Breakdown")
print(df_summary[["model_name", "pooled_r2", *year_cols]].to_markdown(index=False))

### Year-by-Year R² Breakdown
| model_name                                                                                                                                                                 |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                                 |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| MLP-2.0 2-Regime-Mixed (val top-5 avg)                                                                                                                                     |    0.800323 |       0.745352 |       0.825612 |       0.832424 |
| MLP 2-Re

## Systematic-Bias Diagnostic (headline)

mlp-1.2/1.3 documented the 96-family's systematic positive test bias (bias² ≈ 10–17 % of MSE); 2.0 got the 54-family under 5 %; 2.1 got the 54-family to 0.8 % but the 96 (13.9 %) and mixed (12.7 %) families still miss the <5 % criterion. 2.2's debias lever for 96: the small-net pool (width 96–320, dropout 0.4–0.6, lr {3e-4, 4e-4, 6e-4, 8e-4} — lr6e-4 never tested before), huber × lr, mixup × lr, 3-layer small nets, and max_epochs {500, 600} probes for the under-trained small nets. Success criterion: per-family median bias²/MSE < 5 % for ALL three families. Backed by `analyze_bias.py`.

In [8]:
if df_bias is not None:
    print("### Per-family median bias^2/MSE share (honest architectures)")
    print("| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
    print("|:-----------|----------:|----------------------:|----------------:|--------------:|")
    for fam, fam_label in fam_labels.items():
        sub = df_bias[(df_bias["family"] == fam) & df_bias["architecture"].isin(HONEST)]
        if sub.empty:
            continue
        print(f"| {fam} | {len(sub)} | {sub['bias2_mse_share'].median():.4f} | "
              f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
    cols = ["family", "config_id", "architecture", "test_r2", "test_rmse", "test_bias", "bias2_mse_share"]
    print("\n### Worst 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share", ascending=False).head(8)[cols].to_markdown(index=False))
    print("\n### Best 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share").head(8)[cols].to_markdown(index=False))
    if df_bias_cl is not None:
        print("\n### Per-cluster median bias^2/MSE share (honest architectures)")
        print("| family     | cluster |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
        print("|:-----------|--------:|----------------------:|----------------:|--------------:|")
        for fam, fam_label in fam_labels.items():
            for cl in (0, 1):
                sub = df_bias_cl[(df_bias_cl["family"] == fam) & (df_bias_cl["cluster"] == cl)
                                 & df_bias_cl["architecture"].isin(HONEST)]
                if sub.empty:
                    continue
                print(f"| {fam} | {cl} | {sub['bias2_mse_share'].median():.4f} | "
                      f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
else:
    print("bias_summary.csv not found — run analyze_bias.py after the sweep.")

### Per-family median bias^2/MSE share (honest architectures)
| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |
|:-----------|----------:|----------------------:|----------------:|--------------:|
| 2regime_96 | 59 | 0.2169 | 0.0243 | 0.7355 |
| 2regime_54 | 82 | 0.0307 | 0.0084 | 0.7815 |
| 2regime_mixed | 66 | 0.0964 | 0.0149 | 0.7786 |

### Worst 8 configs by bias^2/MSE share (all architectures)
| family        | config_id                          | architecture   |   test_r2 |   test_rmse |   test_bias |   bias2_mse_share |
|:--------------|:-----------------------------------|:---------------|----------:|------------:|------------:|------------------:|
| 2regime_96    | w96x96x96_d0.4_lr1e-3              | mlp            |  0.684285 |   0.0572378 |   0.0323514 |          0.319462 |
| 2regime_96    | w256x256_d0.4_huber0.1_lr6e-4      | mlp            |  0.711224 |   0.0547414 |   0.030543  |          0.311309 |
| 2regime_96    | w320x320_d0.5_hub

## FeatureGroupedMLP / PLR — documented negatives, not re-run in 2.2

2.0 established that the grouped-tower (`fg`, best 0.782) and PLR-encoding (`plr`, best 0.720) architectures underperform the plain MLP (0.790) at this scale — the winning lever was the *feature allocation* (the `2regime_mixed` family), not the tower structure. Per the no-re-spend rule, **2.2 runs no fg/plr configs**; the classes and the validated semantic grouping remain available in `mlp22/feature_groups.py`. The grouping table for the union of the three families' features is printed for reference.

In [9]:
import sys as _sys
_sys.path.insert(0, str(EXP_DIR))
from mlp22.feature_groups import summary_table
data = selected_meta
union_feats = sorted(set(selected_meta.get("shared_backbone_54", [])) | set(selected_meta.get("candidate_pool_96", [])) | set(selected_meta.get("cluster_1_delta_features", [])))
print(f"Union of the 3 families' features: {len(union_feats)}")
print(summary_table(list(union_feats)))

Union of the 3 families' features: 116
| group_id | group | n_features | features |
|---|---|---|---|
| 0 | smap | 26 | A_d_SMAP_sm_interp_kobs14, A_d_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs14, A_grad_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs7, C_lag_SMAP_sm_interp_kobs12, C_lag_SMAP_sm_interp_kobs30, SMAP_ampm_diff_interp, SMAP_sm_am_interp, SMAP_sm_am_interp_lag1, SMAP_sm_am_interp_lag30, SMAP_sm_am_interp_rollrange30, SMAP_sm_am_interp_rollrange7, SMAP_sm_interp_lag7, SMAP_sm_interp_rollrange30, SMAP_sm_interp_rollrange7, SMAP_sm_pm_interp, SMAP_sm_pm_interp_lag1, SMAP_sm_pm_interp_lag30, SMAP_sm_pm_interp_lag7, SMAP_sm_pm_interp_rollmean30, SMAP_sm_pm_interp_rollrange30, SMAP_sm_pm_interp_rollrange7, V_ema_SMAP_sm_interp_kobs30, V_rollmin_SMAP_sm_interp_kobs14, V_rollmin_SMAP_sm_interp_kobs30 |
| 1 | optical | 7 | A_grad_s2_b11_kobs30, V_rollmin_s2_b11_kobs14, V_rollmin_s2_b11_kobs30, V_rollmin_s2_b12_kobs30, V_rollrng_s2_b11_kobs30, s2_b4, s2_b8 |
| 2 | vegetatio

## Early-Stopping Replay (patience-60 re-check, tag 21)

Offline replay of honest epoch-selection rules on the saved per-epoch curves (`analyze_stopping.py --tag 21`). 1.2/1.3 established that patience-60 is the best honest rule; the `swa_val` rule rows are replayed for completeness (no SWA configs run in 2.2 — the curves are the live ones). 2.2 re-checks patience-60 on the new grids, including the max_epochs {500, 600} probes (does extending the cap help the 96 small nets?).

In [10]:
if df_stop21_agg is not None:
    print("### Stopping-rule aggregates (mean pooled test RMSE; lower is better; oracle = unreachable bound)")
    print(df_stop21_agg.to_markdown(index=False))
else:
    print("stopping_21_aggregate.csv not found — run analyze_stopping.py --tag 21 after the sweep.")

### Stopping-rule aggregates (mean pooled test RMSE; lower is better; oracle = unreachable bound)
| family        | rule             |   mean_test_rmse |   median_test_rmse |   n |
|:--------------|:-----------------|-----------------:|-------------------:|----:|
| 2regime_96    | patience60       |        0.0531403 |          0.0532437 | 144 |
| 2regime_96    | patience20       |        0.0531403 |          0.0532437 | 144 |
| 2regime_96    | patience40       |        0.0531403 |          0.0532437 | 144 |
| 2regime_96    | val_aux          |        0.0556548 |          0.0556567 | 144 |
| 2regime_96    | swa_val          |        0.0531403 |          0.0532437 | 144 |
| 2regime_96    | plateau_w20e1e-4 |        0.0713855 |          0.0696428 | 144 |
| 2regime_96    | plateau_w40e1e-4 |        0.0639534 |          0.0600721 | 144 |
| 2regime_96    | plateau_w40e3e-4 |        0.0639534 |          0.0600721 | 144 |
| 2regime_96    | plateau_w60e1e-4 |        0.0559729 |          0.05279

## Extrapolation (OOD) Check

588/6,620 test rows (8.9 %) are OOD on ≥1 top-10 gain feature (same definition as mlp-1.0–2.1). The pure-96 family keeps its OOD strength; the mixed family is in-distribution-strong but OOD-weak (its c1 = 54+10 half carries the 54-family's weak OOD). The 2.2 winners' OOD behavior is reported for the record (family allocation is pinned, so this is a tracking table, not a target).

In [11]:
if df_ood is not None:
    print("### Extrapolation check (OOD test slices)")
    print(df_ood.to_markdown(index=False))
else:
    print("ood_summary.csv not found — run analyze_extrapolation.py after the sweep.")

### Extrapolation check (OOD test slices)
| model                                                   | slice           |    n |       r2 |      rmse |        bias |       mae |
|:--------------------------------------------------------|:----------------|-----:|---------:|----------:|------------:|----------:|
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)               | all             | 6620 | 0.759483 | 0.0499584 |  0.0188537  | 0.0387226 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)               | in_distribution | 6032 | 0.753545 | 0.0514519 |  0.0213449  | 0.0401715 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)               | ood             |  588 | 0.757848 | 0.0306936 | -0.0067026  | 0.0238591 |
| MLP 2regime-96 (5-seed champ)                           | all             | 6620 | 0.75656  | 0.050261  |  0.0204007  | 0.0391776 |
| MLP 2regime-96 (5-seed champ)                           | in_distribution | 6032 | 0.749983 | 0.0518224 |  0.0227929  | 0.0407259 |
| MLP 2regime-96 (5-

## Overfitting-Symptom Analysis

From the saved artifacts (no retraining), via `analyze_overfitting.py`: train-fit vs held-out gap (aux2020 = train-fit), capacity vs test transfer, and the per-epoch curve shape for each family's val winner. 2.1's winners show the familiar pattern (test min early, val flat, train-fit improving); the 2.2 mitigations in play are the denser multi-seed selection and (for the 96-family) the lr6e-4 / max_epochs small-net levers.

In [12]:
print("### Overfitting symptoms (analyze_overfitting.py)")
# The CLI writes overfitting_summary.csv; recompute the key tables inline.
import sys as _sys
if str(EXP_DIR) not in _sys.path:
    _sys.path.insert(0, str(EXP_DIR))
from analyze_overfitting import compute_overfitting
r = compute_overfitting(df_sweep, EXP_DIR)
print("\n#### 1. Train-fit vs held-out gap (median RMSE)")
print("| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |")
print("|:-----------|----------------------:|------:|-------:|------------------:|")
for fam in fam_labels:
    print(f"| {fam} | {r[f'{fam}_med_aux']:.4f} | {r[f'{fam}_med_val']:.4f} | "
          f"{r[f'{fam}_med_test']:.4f} | {r[f'{fam}_train_val_ratio']:.1f}x |")
print("\n#### 2. Capacity vs test transfer (median by n_params bucket)")
print("| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |")
print("|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|")
for fam in fam_labels:
    for row in r[f"{fam}_capacity"]:
        print(f"| {row['family']} | {row['capacity']} | {row['n_configs']} | "
              f"{row['med_val_rmse']:.4f} | {row['med_test_r2']:.4f} | {row['med_test_bias']:.4f} |")
print("\n#### 3. Per-epoch curve shape for the val winner (cluster-0 specialist)")
print("| family     | config_id |   aux_ep100 |   aux_ep260 |   val_plateau |   test_min |   test_min_epoch |   test_at_best_val |   test_final |   test_rise_after_min |")
print("|:-----------|:----------|------------:|------------:|--------------:|-----------:|-----------------:|-------------------:|-------------:|----------------------:|")
for row in r["curve_rows"]:
    print(f"| {row['family']} | {row['config_id']} | {row['aux_ep100']:.4f} | {row['aux_ep260']:.4f} | "
          f"{row['val_plateau']:.4f} | {row['test_min']:.4f} | {row['test_min_epoch']} | "
          f"{row['test_at_best_val']:.4f} | {row['test_final']:.4f} | {row['test_rise_after_min']:.4f} |")
print("\n#### 4. Systematic bias on test (MLP vs XGBoost references)")
for fam in fam_labels:
    print(f"MLP {fam} median test bias: {r[f'{fam}_med_test_bias']:.4f}")
print("XGBoost references (eval-1.1): 2-regime 0.0065, global 0.0105")

### Overfitting symptoms (analyze_overfitting.py)

#### 1. Train-fit vs held-out gap (median RMSE)
| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |
|:-----------|----------------------:|------:|-------:|------------------:|
| 2regime_96 | 0.0357 | 0.0517 | 0.0524 | 1.4x |
| 2regime_54 | 0.0228 | 0.0576 | 0.0476 | 2.5x |
| 2regime_mixed | 0.0217 | 0.0490 | 0.0479 | 2.3x |

#### 2. Capacity vs test transfer (median by n_params bucket)
| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |
|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|
| 2regime_96 | <200k | 48 | 0.0519 | 0.7356 | 0.0240 |
| 2regime_96 | 200-500k | 10 | 0.0505 | 0.7269 | 0.0284 |
| 2regime_96 | 1M+ | 1 | 0.0485 | 0.7595 | 0.0189 |
| 2regime_54 | <200k | 22 | 0.0585 | 0.7870 | 0.0066 |
| 2regime_54 | 200-500k | 42 | 0.0576 | 0.7835 | 0.0088 |
| 2regime_54 | 500k-1M | 14 | 0.0556 | 0.7718 | 0.0092 |
| 2regime_54 | 

The sweep is sized to spend ~1.25 h of the 2 h `gpu_debug` H100 wall allocation: 191 phase-1 + 201 phase-2 + 108 phase-3 + 8 champion job-seeds ≈ 508 jobs at 8 parallel workers (2.1's per-seed mean was 45 s at ~5.9 effective workers).

In [13]:
print("### Timing (H100 PCIe 80 GB, 8 parallel workers)")
print(f"Total sweep wall time: {timing_log.get('sweep_wall_s', float('nan')):.1f} s")
print(f"Total training time (all jobs, GPU-seconds): {sum(j.get('train_time_s', 0.0) for j in timing_log.get('jobs', {}).values()):.0f} s")
print(f"Eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
print()
slow = sorted(timing_log.get("jobs", {}).items(), key=lambda kv: -(kv[1].get("train_time_s") or 0))[:5]
print("Slowest jobs (3-seed config train_time_s):")
for k, v in slow:
    print(f"  {k:55s} {v.get('train_time_s', float('nan')):8.1f}s  n_seeds={v.get('n_seeds')}")


### Timing (H100 PCIe 80 GB, 8 parallel workers)
Total sweep wall time: 3774.5 s
Total training time (all jobs, GPU-seconds): 21888 s
Eval wall time: 8.5 s

Slowest jobs (3-seed config train_time_s):
  2regime_mixed/w512x512x512_d0.3_huber0.15_lr4e-4           163.3s  n_seeds=3
  2regime_54/w384x384x384_d0.3_huber0.1_gelu                 160.8s  n_seeds=3
  2regime_96/w128x128_d0.5_me500                             160.2s  n_seeds=2
  2regime_96/w128x128x128_d0.4_lr6e-4                        153.4s  n_seeds=3
  2regime_mixed/w512x512x512_d0.3_huber0.1                   152.9s  n_seeds=3


## Key Takeaways

1. **The val-selected winners are again below 2.0's 0.8003 — but the sweep
   found the strongest single MLP of the whole 1.0–2.2 series on test.**
   Honest 3-seed val winners: mixed `w512x512x512_d0.3_huber0.03_lr1e-3`
   → test 0.7809 (below 2.1's 0.7844), 54 `w448x448x448_d0.3_huber0.1_gelu_lr1e-3`
   → 0.7596 (below 2.1's 0.7713), 96 unchanged 0.7595; 2.2's mixed val top-5
   ensemble 0.7850 and cross-family 0.7885 both sit below 2.0's 0.8003 /
   0.7932. Meanwhile the **test-best single is the 54-family
   `w320x320_d0.4_huber0.2_gelu_lr6e-4` → 0.7973** (3 seeds, val rank 49/82!),
   with `w320x320_d0.4_gelu_lr6e-4` 0.7960 and mixed `w512x512x512_d0.3_huber0.1_gelu`
   0.7928 (the untested gelu-512³ cell) close behind. The 320²-hubergelu/lr6e-4
   cell is the new frontier — and it is invisible to the val selector.
2. **Val-year diagnostic (NEW): the official val split's 2022 half is the
   noise source for the 54/96 families.** Spearman(val-2021, test) = +0.747
   (96, p=1e-11) / +0.454 (54) while val-2022 = +0.106 / +0.133 (both ns) —
   the full-val mean dilutes the reliable 2021 signal. Selecting on val-2021
   only would pick the 54 `w320x320_d0.2_huber0.1_gelu_lr6e-4` (test 0.7810)
   over the full-val winner (0.7596). The mixed family is different: BOTH
   years are negatively correlated with test (-0.249 / -0.352) — its val-noise
   is structural (the c0 = 96-pool half), not a year artifact. Diagnostic
   only; the deployed rule is unchanged.
3. **54-family selection signal flipped positive: +0.582 at 3 seeds (2.1:
   −0.555)** — the denser seed coverage and the new pool fixed the direction,
   yet the 2→3-seed flip still moved the 54 winner to a worse-test config
   (0.7772 → 0.7596): per-config seed noise and the 3-layer-val-overfit
   pattern (the 54 val top-10 is dominated by 3-layer configs that fail on
   test) remain.
4. **Debias: 54 met (median 3.1 %), mixed improved (12.7 → 9.6 %) but still
   >5 %; 96 worsened (13.9 → 21.7 %).** The mid-lr {4e-4, 6e-4, 8e-4} small-net
   configs are more biased than the lr3e-4 anchor (w256x256_d0.5: 1.1 %), and
   the max_epochs {500, 600} probes did not help (me500 → 0.7682 vs the
   400-cap 0.7834). The 96 criterion remains unmet — capacity control without
   lr3e-4 does not debias this family.
5. **The deliverables that hold up:** the val-year diagnostic (the first
   structural explanation of the val-noise), the 54 `w320x320_d0.4_huber0.2_gelu_lr6e-4`
   test-best reference (0.7973), the 3/3 bit-identical anchors vs 2.1 (max|diff|
   = 0 on a different node), and a fully reproducible v9 sweep (191 configs,
   508 job-seeds, 63 min sweep, 1:06:53 total wall — inside the ~1.25 h target).

All numbers above are the stdout of this notebook; weights/checkpoints/test
predictions under `models/`; preprocessed tensors and per-job logs under
`artifacts/`; figures at the experiment root. See README.md for the full
reproducibility checklist and caveats.
